# Baseline v4.2 — Fine-tuned Bi-Encoder với Dữ Liệu Augmented

## Điểm khác biệt so với v4

| | v4 (gốc) | **v4.2 (augmented)** |
|--|--|--|
| Training pairs | ~6,615 | **~13,117** (+98%) |
| Queries/chunk | ~4 (factual) | ~8 (factual + tình huống + có-không) |
| Data source | `train.jsonl` (round 1) | `train_merged.jsonl` (round 1 + round 2) |
| Model output | `legal_hf_finetuned/` | **`legal_hf_v4_2_augdata/`** |
| FAISS index | `faiss_v4.index` | **`faiss_v4_2.index`** |

## Cải tiến dữ liệu (Data Augmentation)

Round 1 (notebook gen gốc) đã sinh **4 câu hỏi factual** cho mỗi chunk:  
> "Quy định về ... là gì?", "... được định nghĩa như thế nào?"

Round 2 (augment_v2_more_queries.ipynb) bổ sung thêm **4 câu hỏi đa dạng phong cách**:
- **Tình huống:** "Nếu tôi ... thì có được không?", "Khi ... thì phải làm gì?"
- **Có/Không:** "... có được phép ... không?", "Có phải ... không?"
- **Điều kiện:** "Điều kiện để ... là gì?"

**Lý thuyết:** Bi-encoder học rằng nhiều cách hỏi khác nhau đều dẫn đến một passage → vector representation phong phú hơn → khả năng generalize tốt hơn.

```
Pipeline v4.2:
  train_merged.jsonl (13,117 pairs — 8 query styles/chunk)
       ↓ MultipleNegativesRankingLoss
  Legal_HF (fine-tuned v4.2) ← bi-encoder mới
       ↓ FAISS top-50
  Cross-Encoder v5 rerank
       ↓
  Metrics (kỳ vọng > v4: R@1=0.5232, v5: R@1=0.5418)
```

| | v4_base | v4+CE(v5) | **v4.2_base** | **v4.2+CE(v5)** |
|--|--|--|--|--|
| Recall@1 | 0.5232 | 0.5418 | ? | ? |
| Recall@3 | 0.6594 | 0.6873 | ? | ? |
| Recall@5 | 0.7337 | 0.7245 | ? | ? |
| MRR@10   | 0.6091 | 0.6307 | ? | ? |

## Cell 0 — Config & Imports

In [ ]:
import json, csv, time
import numpy as np
import faiss
import torch
import gc
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample, losses

# ── Paths ──
ROOT         = Path(".")
DATA_DIR     = ROOT / "data"
EVAL_DIR     = ROOT / "outputs" / "eval"
TMP_DIR      = ROOT / "outputs" / "tmp"
MDL_DIR      = ROOT / "outputs" / "models"

# ── Input data: dùng file augmented (train_merged.jsonl) ──
# Đặt train_merged.jsonl (từ Google Drive) vào thư mục data/
TRAIN_FILE    = DATA_DIR / "train_merged.jsonl"       # ~13,117 pairs (augmented)
DEV_FILE      = DATA_DIR / "dev.jsonl"                # dev set gốc (không đổi)
TRAIN_NEG     = DATA_DIR / "train_merged_with_neg.jsonl"  # ~26,234 dòng (pos+neg)
EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"

# ── Output paths v4.2 (TÁCH BIỆT với v4 để có thể so sánh) ──
FT_MODEL_DIR    = MDL_DIR  / "legal_hf_v4_2_augdata"      # bi-encoder v4.2
FAISS_INDEX_V42 = TMP_DIR  / "faiss_v4_2.index"
FAISS_MAP_V42   = TMP_DIR  / "faiss_mapping_v4_2.jsonl"
RERANK_CSV_V4   = EVAL_DIR / "rerank_metrics_v4.csv"       # v4 kết quả cũ (để so sánh)
RERANK_CSV_V42  = EVAL_DIR / "rerank_metrics_v4_2.csv"     # v4.2 kết quả mới

# ── Cross-Encoder: dùng v5 (hard-neg trained với v4 bi-encoder) ──
CE_MODEL_PATH = MDL_DIR / "ce_hard_neg_v5" / "final"
if not CE_MODEL_PATH.exists():
    CE_MODEL_PATH = MDL_DIR / "ce_hard_neg_v5"
if not CE_MODEL_PATH.exists():
    # fallback sang CE v1
    CE_MODEL_PATH = MDL_DIR / "cross_encoder_v1" / "saved_model"
    if not CE_MODEL_PATH.exists():
        CE_MODEL_PATH = MDL_DIR / "cross_encoder_v1"

# ── Fine-tuning config (giữ nguyên như v4) ──
BASE_BI_MODEL = "Quockhanh05/Vietnam_legal_embeddings"
FT_EPOCHS     = 5        # 5 epochs như v4
FT_BATCH      = 32       # batch 32 → 31 in-batch negatives mỗi step
FT_LR         = 2e-5     # LR như v4
WARMUP_RATIO  = 0.1
MAX_SEQ_LEN   = 256

# ── Eval config ──
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
ENCODE_BATCH  = 64 if DEVICE == "cuda" else 16
CE_BATCH      = 32
TOP_N         = 50

FT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch     : {torch.__version__}")
print(f"Device    : {DEVICE}")
print(f"Base model: {BASE_BI_MODEL}")
print(f"FT Epochs : {FT_EPOCHS}, Batch: {FT_BATCH}, LR: {FT_LR}")
print(f"Train file: {TRAIN_FILE}")
print(f"CE model  : {CE_MODEL_PATH}")
print(f"Output    : {FT_MODEL_DIR}")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows: break
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: errors += 1
    if errors: print(f"  ⚠ {errors} malformed lines in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_hit(faiss_id, expected_citations, mapping):
    row = mapping[faiss_id]
    for ec in expected_citations:
        ci = ec.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if (row["van_ban"] == ec.get("van_ban", "") and
            row["dieu"]    == ec.get("dieu",    "") and
            row["khoan"]   == ec.get("khoan",   "")): return True
    return False

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

print("Utilities loaded ✓")

## Cell 2 — Chuẩn bị dữ liệu fine-tuning

> **v4.2:** Dữ liệu gấp đôi v4 — 13,117 pairs với 8 phong cách câu hỏi khác nhau/chunk  
> Dùng `MultipleNegativesRankingLoss` → in-batch negatives tự động (batch=32 → 31 negatives/query)

In [ ]:
# Thu thập (query, positive_passage) từ train_merged.jsonl
# Lưu ý: TRAIN_NEG (train_merged_with_neg) cũng chứa positives (label=1)
seen_pairs = set()
train_examples = []

for filepath in [TRAIN_FILE, TRAIN_NEG]:
    for r in load_jsonl(filepath):
        if r.get("label") != 1: continue       # chỉ lấy positives
        q = r.get("query", "").strip()
        p = r.get("passage", "").strip()
        if not q or not p: continue
        key = (q, p)
        if key in seen_pairs: continue
        seen_pairs.add(key)
        train_examples.append(InputExample(texts=[q, p]))

print(f"Fine-tuning pairs: {len(train_examples)}")
print(f"  (v4 có 6,615 pairs — v4.2 tăng {len(train_examples) - 6615:+,} pairs)")
print(f"")
print(f"Example:")
ex = train_examples[0]
print(f"  Query  : {ex.texts[0][:80]}...")
print(f"  Passage: {ex.texts[1][:80]}...")
print()
print(f"⚡ In-batch negatives per step: {FT_BATCH - 1} negatives/query")
print(f"⚡ Total steps/epoch: {len(train_examples)//FT_BATCH}")
print(f"⚡ Warmup steps: {int(len(train_examples)/FT_BATCH * FT_EPOCHS * WARMUP_RATIO)}")

## Cell 3 — Fine-tune Legal_HF Bi-Encoder (v4.2)

> ⏱️ Ước tính thời gian với 13,117 pairs:
> - RTX 3050 Ti (4GB): ~**10-20 phút/epoch** → 5 epochs ≈ **50-100 phút**
> - (v4 mất ~25-50 phút vì chỉ có 6,615 pairs)
>
> 💡 Nếu OOM: Giảm `FT_BATCH = 16` ở Cell 0.

In [ ]:
# Load model gốc (cùng base như v4)
print(f"Loading base model: {BASE_BI_MODEL}")
bi_model = SentenceTransformer(BASE_BI_MODEL, device=DEVICE)
bi_model.max_seq_length = MAX_SEQ_LEN
print(f"Model loaded ✓ | dim={bi_model.get_sentence_embedding_dimension()} | max_seq={MAX_SEQ_LEN}")

# DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=FT_BATCH)

# Loss
train_loss = losses.MultipleNegativesRankingLoss(bi_model)

warmup_steps = int(len(train_dataloader) * FT_EPOCHS * WARMUP_RATIO)
total_steps  = len(train_dataloader) * FT_EPOCHS
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

# Fine-tune
print(f"\nStarting fine-tuning v4.2: {FT_EPOCHS} epochs...")
t0 = time.time()

bi_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=FT_EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": FT_LR},
    use_amp=(DEVICE == "cuda"),       # mixed precision trên GPU
    show_progress_bar=True,
    checkpoint_path=str(FT_MODEL_DIR),
    checkpoint_save_steps=len(train_dataloader),  # save mỗi epoch
    checkpoint_save_total_limit=2,
)

elapsed = round((time.time()-t0)/60, 1)
print(f"\nFine-tuning done in {elapsed} min")

# Save final model
final_model_path = FT_MODEL_DIR / "final"
bi_model.save(str(final_model_path))
print(f"Model saved → {final_model_path}")

# Save config
ft_config = {
    "version": "v4.2",
    "base_model": BASE_BI_MODEL,
    "final_model": str(final_model_path),
    "data_augmentation": {
        "round1": "train.jsonl (6,615 pairs, 4 factual queries/chunk)",
        "round2": "train_aug2.jsonl (6,563 pairs, 4 situational queries/chunk)",
        "merged": "train_merged.jsonl (13,117 unique pairs — +98% vs v4)"
    },
    "epochs": FT_EPOCHS, "batch_size": FT_BATCH,
    "lr": FT_LR, "warmup_ratio": WARMUP_RATIO,
    "max_seq_length": MAX_SEQ_LEN,
    "train_pairs": len(train_examples),
    "training_minutes": elapsed,
}
(FT_MODEL_DIR / "ft_config.json").write_text(
    json.dumps(ft_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Config saved ✓")

## Cell 4 — Encode Corpus + Build FAISS (v4.2)

In [ ]:
# Giải phóng VRAM
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

# Load fine-tuned model
final_model_path = FT_MODEL_DIR / "final"
print(f"Loading fine-tuned model: {final_model_path}")
ft_bi = SentenceTransformer(str(final_model_path), device=DEVICE)
print(f"Fine-tuned model loaded ✓ | dim={ft_bi.get_sentence_embedding_dimension()}")

# Thu thập corpus từ TẤT CẢ file data
# (bao gồm cả train_merged để corpus đầy đủ)
seen_passages = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage", "")
        if p and p not in seen_passages:
            meta = r.get("meta", {})
            seen_passages[p] = {
                "passage":     p,
                "chunk_index": meta.get("chunk_index", -1),
                "van_ban":     meta.get("van_ban",  ""),
                "chuong":      meta.get("chuong",   ""),
                "dieu":        meta.get("dieu",     ""),
                "khoan":       meta.get("khoan",    ""),
                "diem":        meta.get("diem",     ""),
            }

corpus = list(seen_passages.values())
texts  = [c["passage"] for c in corpus]
print(f"Corpus: {len(corpus)} unique passages")

# Encode với fine-tuned model v4.2
print("Encoding corpus with fine-tuned v4.2 model...")
t0 = time.perf_counter()
embeddings = ft_bi.encode(
    texts, batch_size=ENCODE_BATCH, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
).astype("float32")
print(f"Shape: {embeddings.shape} | Time: {time.perf_counter()-t0:.1f}s")

# FAISS
index_v42 = faiss.IndexFlatIP(embeddings.shape[1])
index_v42.add(embeddings)
faiss.write_index(index_v42, str(FAISS_INDEX_V42))
print(f"FAISS → {FAISS_INDEX_V42}")

mapping_v42 = [{"faiss_id": i, **c} for i, c in enumerate(corpus)]
write_jsonl(FAISS_MAP_V42, mapping_v42)
print(f"Mapping → {FAISS_MAP_V42} ({len(mapping_v42)} entries)")

## Cell 5 — Evaluate v4.2 Baseline + v4.2 + Rerank (CE v5)

In [ ]:
# Giải phóng bi-encoder, load cross-encoder v5
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

print(f"Loading cross-encoder: {CE_MODEL_PATH}")
ce_model = CrossEncoder(str(CE_MODEL_PATH), max_length=256, device=DEVICE)
print("Cross-encoder loaded ✓")

eval_qa = load_jsonl(EVAL_QA_FILE)
print(f"Eval QA: {len(eval_qa)} questions")

r_base   = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}
r_rerank = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}

for item in tqdm(eval_qa, desc="Evaluate v4.2"):
    query = item["query"]
    ec    = item["expected_citations"]

    # Retrieve với fine-tuned bi-encoder v4.2
    q_emb = ft_bi.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, ids = index_v42.search(q_emb, TOP_N)
    ids    = ids[0].tolist()

    # Baseline metrics
    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit(i,ec,mapping_v42) for i in ids[:k] if i>=0) else 0)
    mrr = 0.0
    for rank, i in enumerate(ids[:10], 1):
        if i >= 0 and is_hit(i, ec, mapping_v42): mrr = 1.0/rank; break
    r_base["MRR@10"].append(mrr)

    # Rerank với CE v5
    cands   = [(mapping_v42[i]["passage"], i) for i in ids if i >= 0]
    rscores = ce_model.predict([[query, c[0]] for c in cands], batch_size=CE_BATCH) if cands else []
    ranked  = sorted(zip(rscores, [c[1] for c in cands]), reverse=True)
    r_ids   = [r[1] for r in ranked]

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_rerank[key].append(1 if any(is_hit(i,ec,mapping_v42) for i in r_ids[:k]) else 0)
    mrr = 0.0
    for rank, i in enumerate(r_ids[:10], 1):
        if is_hit(i, ec, mapping_v42): mrr = 1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

print("\n── v4.2 Results ──")
print(f"  {'Metric':<10} {'v4.2 Baseline':>14} {'v4.2+CE(v5)':>14} {'Δ':>8}")
print("  " + "-"*50)
for k, key in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    b  = avg(r_base[k])
    re = avg(r_rerank[k])
    print(f"  {key:<10} {b:>14.4f} {re:>14.4f} {re-b:>+8.4f}")

## Cell 6 — So sánh đầy đủ v4 vs v4.2 & Lưu CSV

In [ ]:
# Đọc v4 kết quả cũ để so sánh
v4 = {}
if RERANK_CSV_V4.exists():
    with open(RERANK_CSV_V4, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            v4[row["metric"]] = {
                "base":   float(row.get("v4_finetuned", 0) or 0),
                "rerank": float(row.get("v4_reranked", 0) or 0),
            }
else:
    # Fallback: hard-code kết quả v4 đã biết
    v4 = {
        "Recall@1": {"base": 0.5232, "rerank": 0.5418},
        "Recall@3": {"base": 0.6594, "rerank": 0.6873},
        "Recall@5": {"base": 0.7337, "rerank": 0.7245},
        "MRR@10":   {"base": 0.6091, "rerank": 0.6307},
    }
    print("⚠ rerank_metrics_v4.csv không tìm thấy → dùng giá trị cứng đã biết")

v42_base   = {"Recall@1":avg(r_base["R@1"]),   "Recall@3":avg(r_base["R@3"]),
              "Recall@5":avg(r_base["R@5"]),   "MRR@10":avg(r_base["MRR@10"])}
v42_rerank = {"Recall@1":avg(r_rerank["R@1"]), "Recall@3":avg(r_rerank["R@3"]),
              "Recall@5":avg(r_rerank["R@5"]), "MRR@10":avg(r_rerank["MRR@10"])}

print("\n" + "="*100)
hdr = f"  {'Metric':<10} {'v4_base':>12} {'v4+CE(v5)':>12} {'v4.2_base':>12} {'v4.2+CE(v5)':>14} {'Δ(4.2R-4R)':>12}"
print(hdr)
print("="*100)
for metric in ["Recall@1","Recall@3","Recall@5","MRR@10"]:
    v4b  = v4.get(metric, {}).get("base",  float("nan"))
    v4r  = v4.get(metric, {}).get("rerank",float("nan"))
    v42b = v42_base[metric]
    v42r = v42_rerank[metric]
    delta= v42r - v4r
    sign = "+" if delta >= 0 else ""
    icon = "✅" if delta > 0 else ("=" if abs(delta) < 0.001 else "❌")
    print(f"  {metric:<10} {v4b:>12.4f} {v4r:>12.4f} {v42b:>12.4f} {v42r:>14.4f} {sign}{delta:>11.4f} {icon}")
print("="*100)

# Lưu CSV
rows_v42 = [
    {"metric":"Recall@1", "v4_baseline":v4.get("Recall@1",{}).get("base",""), "v4_reranked":v4.get("Recall@1",{}).get("rerank",""), "v4_2_baseline":v42_base["Recall@1"], "v4_2_reranked":v42_rerank["Recall@1"]},
    {"metric":"Recall@3", "v4_baseline":v4.get("Recall@3",{}).get("base",""), "v4_reranked":v4.get("Recall@3",{}).get("rerank",""), "v4_2_baseline":v42_base["Recall@3"], "v4_2_reranked":v42_rerank["Recall@3"]},
    {"metric":"Recall@5", "v4_baseline":v4.get("Recall@5",{}).get("base",""), "v4_reranked":v4.get("Recall@5",{}).get("rerank",""), "v4_2_baseline":v42_base["Recall@5"], "v4_2_reranked":v42_rerank["Recall@5"]},
    {"metric":"MRR@10",   "v4_baseline":v4.get("MRR@10",{}).get("base",  ""), "v4_reranked":v4.get("MRR@10",{}).get("rerank",  ""), "v4_2_baseline":v42_base["MRR@10"],   "v4_2_reranked":v42_rerank["MRR@10"]},
]
with open(RERANK_CSV_V42, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","v4_baseline","v4_reranked","v4_2_baseline","v4_2_reranked"])
    w.writeheader(); w.writerows(rows_v42)

print(f"\nSaved → {RERANK_CSV_V42} ✓")